# Prediction Visualization

This notebook visualizes model predictions on test images.


In [ ]:
import sys
from pathlib import Path
import torch

# Add parent directory to path
sys.path.append(str(Path().resolve().parent))

from src.visualization import visualize_predictions
from src.training import create_model
from src.preprocessing import VOCDataset, get_transforms
from config.config import MODELS_DIR, MODEL_CONFIG, RESULTS_DIR, TRAIN_CONFIG


## Load Model


In [ ]:
# Load model
device = torch.device(TRAIN_CONFIG['device'] if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create model
model = create_model(num_classes=MODEL_CONFIG['num_classes'])
model.to(device)

# Load weights
model_path = MODELS_DIR / "best_model.pth"
if model_path.exists():
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"Model loaded from {model_path}")
else:
    print(f"Warning: Model not found at {model_path}")
    print("Using untrained model for visualization")


## Visualize Predictions on Validation Set


In [ ]:
# Create validation dataset
val_dataset = VOCDataset(split='val', transform=get_transforms(train=False))

# Visualize predictions
save_dir = RESULTS_DIR / "visualizations" / "predictions"
visualize_predictions(
    model=model,
    dataset=val_dataset,
    device=device,
    num_images=10,
    score_threshold=0.5,
    save_dir=save_dir
)


## Visualize Predictions on Training Set (Sample)


In [ ]:
# Create training dataset
train_dataset = VOCDataset(split='train', transform=get_transforms(train=False))

# Visualize predictions
visualize_predictions(
    model=model,
    dataset=train_dataset,
    device=device,
    num_images=5,
    score_threshold=0.5,
    save_dir=save_dir
)
